# **Demo Nootebook of Bottleneck Adapters (PE Transfer Learning for NLP)**

This notebook was created for demonstration purposes as part of the seminar *Adapting and Fine-Tuning Foundation Models*. The characteristics of the bottleneck architecture will be examined from a practical perspective, and an application example will be demonstrated. The following libraries were used for this notebook and the code implemented within it:
- **[Hugging Face Transformers](https://huggingface.co/docs/transformers/index)**
- **[AdapterHub (Adapters)](https://docs.adapterhub.ml/)**
- **[PyTorch](https://pytorch.org/)**

First, the base model and the properties of the added bottleneck adapters are examined. Subsequently, the use of adapters is demonstrated using two different text classification tasks.

### 1. RoBERTa + Bottleneck Adapters

First, we load a pre-trained version of the **[RoBERTa transformer](https://huggingface.co/docs/transformers/model_doc/roberta)**.

In [72]:
from transformers import AutoTokenizer
from adapters import AutoAdapterModel
import torch

model_name = "roberta-base"
base_model = AutoAdapterModel.from_pretrained(model_name)
base_parameters = sum(p.numel() for p in base_model.parameters())
base_parameters_trainable = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
print(f"Roberta-base parameters: {base_parameters/ 1e6:.2f}M")
print(f"-- Number of Transformer-blocks: {len(base_model.roberta.encoder.layer)}")
print(f"-- Number of trainable params: {base_parameters_trainable/1e6:.2f}M")

Some weights of RobertaAdapterModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['heads.default.3.bias', 'roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Roberta-base parameters: 125.29M
-- Number of Transformer-blocks: 12
-- Number of trainable params: 125.29M


Now we want to add an adapter for a specific down-stream tasks and showcase necessary steps for later training.

In [66]:
adapter_name = "demo"
adapted_model = AutoAdapterModel.from_pretrained(model_name)
adapted_model.add_adapter(adapter_name, config="houlsby")
adapted_model.train_adapter(adapter_name)


adapted_model_parameters = sum(p.numel() for p in adapted_model.parameters())
adapter_parameters = sum(p.numel() for (name, p) in adapted_model.roberta.encoder.named_parameters() if adapter_name in name)
adapted_model_parameters_trainable = sum(p.numel() for p in adapted_model.roberta.encoder.parameters() if p.requires_grad)
# Showcasing some statistics about the adapter
print(f"Adapted-model parameters: {adapted_model_parameters/ 1e6:.2f}M")
print(f"-- Paramater growth: {adapted_model_parameters / base_parameters:.2f}%")
print(f"-- Adapter parameter: {adapter_parameters /1e6:.2f}M")
print(f"-- Trainable params: {adapted_model_parameters_trainable/1e6:.2f}M")


Some weights of RobertaAdapterModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['heads.default.3.bias', 'roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
There are adapters available but none are activated for the forward pass.


Adapted-model parameters: 127.08M
-- Paramater growth: 1.01%
-- Adapter parameter: 1.79M
-- Trainable params: 1.79M


Finally, we want to see how the adapter looks and how it resembles a bottleneck architecture.

In [97]:
import difflib
from IPython.display import display, HTML

base_transformer_block = str(base_model.roberta.encoder.layer[-1])
adapted_transformer_block = str(adapted_model.roberta.encoder.layer[-1])
differ = difflib.HtmlDiff(wrapcolumn=150)
html_output = differ.make_file(
    base_transformer_block.splitlines(),
    adapted_transformer_block.splitlines(),
    fromdesc="Base-Transformer block",
    todesc="Adapted-Transformer block",
    context=True,
    numlines=3
)
display(HTML(html_output))

The output of the upper cell displays the properties of the bottleneck projection and subsequent non-linear activation.

**Note**: The implementation differs from the adapters proposed from **[Houlsby et al.](https://arxiv.org/abs/1902.00751)** in two ways:
- The layer normalizations are not set to trainable per default. This can be evidenced by the following computation:
    - each bottleneck projections holds (2x768x48 weights) + (768 + 48 biases) = 74544
    - each encoder layer gets two adapters and there are 12 encoder-layers, resulting in 12*2*74544 = 1.789.056 Parameters
    - there are no trainable parameters "left" to be laying in LayerNorms
- While the others experimented with ReLU/GELU, this implementation uses SiLU as non-linear activation

### 2. Demo: Sarcasm and Seminar Adapter

For the demonstration, two adapters + classification heads were trained in adavance for the following tasks:
- Sarcasm Adapter: Is used while classifying given sentences in non-irony ("0") and irony("1") trained on **[tweet_eval](https://huggingface.co/datasets/cardiffnlp/tweet_eval)**.
- Seminar Adapter: Is used to classify arxiv papers by their abstract into classes non-releveant ("0") and relevant ("1") for our seminar.

- **Note**: Since preparing data based on their relevance would be very time-consuming ,the following classifcation scheme was adopted for the arXiv shortcuts listed below:
    * 0: non-releveant papers, scraped by:
        * "cs.AR",  # (Hardware Architecture)
        * "cs.CR",  # (Cryptography and Security)
        * "cs.CV",  # (Computer Vision and Pattern Recognition)
        * "cs.IR",  # (Information Retrieval)
         * "cs.DB",  # (Databases)
    * 1: relevant papers, scraped by:
        * "cs.LG", # Machine Learning,
        * "cs.CL", # Computation and Language

Both adapters (including their class. heads) have been trained for 5 epochs with the same train config:
- batch_size=16
- learning_rate=1e-4
- optimizer="adamw_torch"

In [95]:
adapter_paths = {
        "sarcasm_adapter": "model/sarcasm_adapter/checkpoint-895/sarcasm_adapter",
        "seminar_adapter": "model/seminar_adapter/checkpoint-1270/seminar_adapter",
    }

def predict(text, model, tokenizer, device):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=1)
    predicted_class_id = torch.argmax(probabilities, dim=1).item()
    confidence = probabilities[0][predicted_class_id].item()

    return predicted_class_id, confidence

#### a) Sarcasm Adapter

First, we will load the model and adapter:

In [101]:
import os
%cd /pfad/zu/deinem/hauptverzeichnis
%pwd
os.chdir("/home/lukas/Schreibtisch/BDL/code/PythonProject")
model = AutoAdapterModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model.active_adapters = "sarcasm_adapter"
model.load_adapter(adapter_paths["sarcasm_adapter"])
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

[Errno 2] No such file or directory: '/pfad/zu/deinem/hauptverzeichnis'
/home/lukas/Schreibtisch/BDL/code/PythonProject/notebook


Some weights of RobertaAdapterModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['heads.default.3.bias', 'roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaAdapterModel(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttentionWithAdapters(
              (query): LoRALinearTorch(
                in_features=768, out_features=768, bias=True
                (shared_parameters): ModuleDict()
                (loras): ModuleDict()
              )
              (key): LoRALinearTorch(
                in_features=768, out_features=768, bias=True
                (shared_parameters): ModuleDict()
                (loras): ModuleDict()
              )
              

Now we will look at inference results for some example sentences.

In [102]:
sentences = [
        "I love doing work and getting nothing out of it.",
        "I like summer and sunshine.",
        "Yeah, don't worry about me, I absolutely love standing alone in the freezing cold for two hours.",
        "The Deutsche Bahn is always on time.",
        "I like dancing.",
        "I like pineapple on pizza."
]
print(f"[START] Starting inference for given sentences...")
print(f"-- Active adapters: {model.active_adapters}")
for sentence in sentences:
    predicted_class_id, confidence = predict(sentence, model, tokenizer, device)
    print(f"[INPUT]: {sentence}")
    if predicted_class_id == 0:
        print(f"✅ I can't hear sarcasm! \n")
    else:
        print(f"❌ This smells like sarcasm! \n")

There are adapters available but none are activated for the forward pass.


[START] Starting inference for given sentences...
-- Active adapters: None


RuntimeError: a Tensor with 50265 elements cannot be converted to Scalar

#### b) Seminar Adapter

We will do the same for the seminar adapter, but load the adapter in the same model instance.

In [103]:
model.load_adapter(adapter_paths["seminar_adapter"])
model.active_adapters = "seminar_adapter"
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

There are adapters available but none are activated for the forward pass.


RobertaAdapterModel(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttentionWithAdapters(
              (query): LoRALinearTorch(
                in_features=768, out_features=768, bias=True
                (shared_parameters): ModuleDict()
                (loras): ModuleDict()
              )
              (key): LoRALinearTorch(
                in_features=768, out_features=768, bias=True
                (shared_parameters): ModuleDict()
                (loras): ModuleDict()
              )
              

Finally, we can let the model predict if an arbitrary paper from arXiv might be relevant for our seminar or not.

In [107]:
papers = [
        {
            "title": "U-Net: Convolutional Networks for Biomedical Image Segmentation",
            "abstract": "There is large consent that successful training of deep networks requires many thousand annotated training samples. In this paper, we present a network and training strategy that relies on the strong use of data augmentation to use the available annotated samples more efficiently. The architecture consists of a contracting path to capture context and a symmetric expanding path that enables precise localization. We show that such a network can be trained end-to-end from very few images and outperforms the prior best method (a sliding-window convolutional network) on the ISBI challenge for segmentation of neuronal structures in electron microscopic stacks. Using the same network trained on transmitted light microscopy images (phase contrast and DIC) we won the ISBI cell tracking challenge 2015 in these categories by a large margin. Moreover, the network is fast. Segmentation of a 512x512 image takes less than a second on a recent GPU."
        },
        {
            "title": "Parameter-Efficient Transfer Learning for NLP",
            "abstract": "Fine-tuning large pre-trained models is an effective transfer mechanism in NLP. However, in the presence of many downstream tasks, fine-tuning is parameter inefficient: an entire new model is required for every task. As an alternative, we propose transfer with adapter modules. Adapter modules yield a compact and extensible model; they add only a few trainable parameters per task, and new tasks can be added without revisiting previous ones. The parameters of the original network remain fixed, yielding a high degree of parameter sharing. To demonstrate adapter's effectiveness, we transfer the recently proposed BERT Transformer model to 26 diverse text classification tasks, including the GLUE benchmark. Adapters attain near state-of-the-art performance, whilst adding only a few parameters per task. On GLUE, we attain within 0.4% of the performance of full fine-tuning, adding only 3.6% parameters per task. By contrast, fine-tuning trains 100% of the parameters per task."
        },
        {
            "title": "Stroke-Based Cursive Character Recognition",
            "abstract": "Human eye can see and read what is written or displayed either in natural handwriting or in printed format. The same work in case the machine does is called handwriting recognition. Handwriting recognition can be broken down into two categories: off-line and on-line. ..."
        },
        {
            "title": "LoRA: Low-Rank Adaptation of Large Language Models",
            "abstract": "An important paradigm of natural language processing consists of large-scale pre-training on general domain data and adaptation to particular tasks or domains. As we pre-train larger models, full fine-tuning, which retrains all model parameters, becomes less feasible. Using GPT-3 175B as an example -- deploying independent instances of fine-tuned models, each with 175B parameters, is prohibitively expensive. We propose Low-Rank Adaptation, or LoRA, which freezes the pre-trained model weights and injects trainable rank decomposition matrices into each layer of the Transformer architecture, greatly reducing the number of trainable parameters for downstream tasks. Compared to GPT-3 175B fine-tuned with Adam, LoRA can reduce the number of trainable parameters by 10,000 times and the GPU memory requirement by 3 times. LoRA performs on-par or better than fine-tuning in model quality on RoBERTa, DeBERTa, GPT-2, and GPT-3, despite having fewer trainable parameters, a higher training throughput, and, unlike adapters, no additional inference latency. We also provide an empirical investigation into rank-deficiency in language model adaptation, which sheds light on the efficacy of LoRA. We release a package that facilitates the integration of LoRA with PyTorch models and provide our implementations and model checkpoints for RoBERTa, DeBERTa, and GPT-2 at"
        }
]
print(f"[START] Starting inference for given papers...")
print(f"-- Active adapters: {model.active_adapters}")
for paper in papers:
    title = paper["title"]
    print(f"\n [START] Starting classification of paper: '{title}' ...")
    predicted_class_id, confidence = predict(paper["abstract"], model, tokenizer, device)
    if predicted_class_id == 1:
        print(f"✅ Prof. Eggensperger would like to read this!")
    else:
        print(f"❌ Given paper seems irrelevant for our seminar.")

[START] Starting inference for given papers...
-- Active adapters: Stack[sarcasm_adapter]

 [START] Starting classification of paper: 'U-Net: Convolutional Networks for Biomedical Image Segmentation' ...
❌ Given paper seems irrelevant for our seminar.

 [START] Starting classification of paper: 'Parameter-Efficient Transfer Learning for NLP' ...
❌ Given paper seems irrelevant for our seminar.

 [START] Starting classification of paper: 'Stroke-Based Cursive Character Recognition' ...
✅ Prof. Eggensperger would like to read this!

 [START] Starting classification of paper: 'LoRA: Low-Rank Adaptation of Large Language Models' ...
❌ Given paper seems irrelevant for our seminar.


**Note**: We used both adapters with the same loaded RoBERTa model. But we have to set the active adapter manually. Setting the active adapter back to sarcasm adapter, the papers are not correctly classified anymore.

In [106]:
model.active_adapters = "sarcasm_adapter"
print(f"[START] Starting inference for given papers...")
print(f"-- Active adapters: {model.active_adapters}")
for paper in papers:
    title = paper["title"]
    print(f"\n [START] Starting classification of paper: '{title}' ...")
    predicted_class_id, confidence = predict(paper["abstract"], model, tokenizer, device)
    if predicted_class_id == 1:
        print(f"✅ Prof. Eggensperger would like to read this!")
    else:
        print(f"❌ Given paper seems irrelevant for our seminar.")

[START] Starting inference for given papers...
-- Active adapters: Stack[sarcasm_adapter]

 [START] Starting classification of paper: 'U-Net: Convolutional Networks for Biomedical Image Segmentation' ...
❌ Given paper seems irrelevant for our seminar.

 [START] Starting classification of paper: 'Parameter-Efficient Transfer Learning for NLP' ...
❌ Given paper seems irrelevant for our seminar.

 [START] Starting classification of paper: 'Stroke-Based Cursive Character Recognition' ...
✅ Prof. Eggensperger would like to read this!

 [START] Starting classification of paper: 'LoRA: Low-Rank Adaptation of Large Language Models' ...
❌ Given paper seems irrelevant for our seminar.
